In [1]:
import pandas as pd
import numpy as np
import polars as pl
import plotly.express as px
import json
import requests
import time
import datetime
import sys
import os
import matplotlib
import pickle
import gc

from tqdm.notebook import tqdm_notebook as tqdm
from pprint import pprint
from io import StringIO

# Settings
pd.options.display.max_rows = 1_000
pd.options.display.max_columns = 100
pd.options.display.width = 20000
pd.set_option('display.float_format', lambda x: '%.4f' % x)
np.set_printoptions(suppress=True)
pl.Config.set_tbl_rows(100)

from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

In [3]:
# https://github.com/vollib/py_vollib
# https://py-vollib-vectorized.readthedocs.io/en/latest/

# import pandas_datareader as pdr
# Install pandas datareader as normal using: pip install pandas-datareader
# Under Lib\site-packages\pandas_datareader\compat, open init.py
# Remove the following import: from distutils.version import LooseVersion
# Add the following import: from packaging.version import Version
# Search(Ctrl+F) for LooseVersion and change to Version

# Get stocks and options data

In [16]:
import pandas_datareader as pdr

start_date = '1950-01-01'
end_date = datetime.datetime.now().strftime('%Y-%m-%d')
treasury_y1 = pdr.get_data_fred('DGS1', start=start_date, end=end_date)
treasury_y1['DGS1'] = treasury_y1['DGS1'].ffill() / 100

treasury_y1 = pl.from_pandas(treasury_y1.reset_index(), schema_overrides={'DATE': pl.Date, 'DGS1': pl.Float32})
treasury_y1 = treasury_y1.rename({col: col.lower() for col in treasury_y1.columns})

In [17]:
# Load stocks data
stocks_df = pl.read_parquet('data/tiingo_stocks_with_features.parquet').with_columns(
    (pl.col('divYieldSum') + 1).log().alias('divYieldSum'),
    (pl.col('divCash') / pl.col('close').shift(1)).alias('divYield')
)

# Load options data
# file_name = 'batch_2022_5'
# options_df = pl.read_parquet(f'data/temp/{file_name}.parquet')\
#     .filter(pl.col('ticker').is_in(set(stocks_df['ticker'])))\
#     .with_columns(
#         pl.col('right').str.to_lowercase().alias('right'),
#         (pl.col('strike') / 1_000).alias('strike'),
#         ((pl.col('expiration') - pl.col('date')).dt.total_days().alias('timeToExp') / 365).cast(pl.Float32)
#     ).with_columns(
#         pl.when(pl.col('timeToExp') == 0).then(1e-3).otherwise(pl.col('timeToExp')).alias('timeToExp')
#     )

In [18]:
options_df = pl.DataFrame()
for c_dir, path, files in os.walk('data\options'):
    need_files = [file for file in files if '_SPY_opts' in file]
    for file in need_files:
        print(file)
        cur_df = pl.read_parquet(os.path.join(c_dir, file))
        options_df = pl.concat([options_df, cur_df])


options_df = options_df\
    .drop('root')\
    .filter(pl.col('ticker').is_in(set(stocks_df['ticker'])))\
    .with_columns(
        expiration=pl.col('expiration').cast(pl.String).str.strptime(pl.Date, format='%Y%m%d')
    ).with_columns(
        pl.col('right').str.to_lowercase().alias('right'),
        (pl.col('strike') / 1_000).alias('strike'),
        ((pl.col('expiration') - pl.col('date')).dt.total_days().alias('timeToExp') / 365).cast(pl.Float32)
    ).with_columns(
        pl.when(pl.col('timeToExp') == 0).then(1e-3).otherwise(pl.col('timeToExp')).alias('timeToExp')
    )

20200102_20201231_SPY_opts.parquet
20210104_20211231_SPY_opts.parquet
20220103_20221230_SPY_opts.parquet
20230103_20231229_SPY_opts.parquet
20240102_20241231_SPY_opts.parquet
20250102_20250124_SPY_opts.parquet


# Create dividends for each option

In [19]:
stocks_div_connect = stocks_df[['date', 'ticker', 'divYield', 'divYieldSum']].filter(pl.col('divYield') > 0)

stocks_threshold = stocks_div_connect\
    .group_by('ticker').agg(
        divyYieldMean=pl.col('divYield').mean(),
        divYieldStd=pl.col('divYield').std()
    )\
    .with_columns((pl.col('divyYieldMean') + pl.col('divYieldStd') * 3).alias('divYieldThreshold'))
stocks_div_connect = stocks_div_connect\
    .join(stocks_threshold.drop('divYieldStd'), how='left', on='ticker')\
    .with_columns(
    pl.when(pl.col('divYieldThreshold') <= pl.col('divYield'))\
        .then(pl.col('divyYieldMean'))\
        .otherwise(pl.col('divYield'))\
        .alias('divYield')
)

options_div_connect = options_df.unique(['expiration', 'date', 'ticker'])

In [20]:
options_div_connect = options_div_connect\
    .join(stocks_div_connect, how='left', on='ticker')\
    .filter(
        (pl.col('date_right') > pl.col('date').dt.offset_by(by='-1y')) & 
        (pl.col('date_right') <= pl.col('expiration').dt.offset_by(by='-1y')) &
        (pl.col('date_right') <= pl.col('date'))
    )\
    .group_by(['ticker', 'expiration', 'date']).agg(divYieldSum=pl.col('divYield').sum().cast(pl.Float32))

options_df = options_df.join(options_div_connect, how='left', on=['ticker', 'date', 'expiration'])\
    .with_columns(pl.col('divYieldSum').fill_null(0))

del stocks_div_connect, options_div_connect
gc.collect()

664

# Calc splits

In [21]:
options_split_connect = options_df.unique(['expiration', 'date', 'ticker'])
split_df = stocks_df[['date', 'ticker', 'splitFactor']].filter(pl.col('splitFactor') != 1)

options_split_connect = options_split_connect.join(
    split_df,
    how='inner', 
    on='ticker'
).filter(
    (pl.col('date_right') > pl.col('date')) & 
    (pl.col('date_right') <= pl.col('expiration'))
).with_columns(
    splitFactor=pl.col('splitFactor').cum_prod().over(['ticker', 'expiration', 'date'])
)

In [22]:
options_df = options_df.join(
    options_split_connect[['ticker', 'date', 'expiration', 'splitFactor']], 
    how='left', 
    on=['ticker', 'date', 'expiration']
).with_columns(pl.col('splitFactor').fill_null(1))

del split_df, options_split_connect
gc.collect()

36

# Calc greeks and IV

In [23]:
options_df = options_df\
    .join(stocks_df[['date', 'ticker', 'close']].rename({'close': 'base_close'}), how='left', on=['date', 'ticker'])\
    .join(treasury_y1, how='left', on='date')\
    .filter(~pl.col('base_close').is_null())\
    .sort('date', 'expiration', 'strike', 'right')\
    .with_row_index("idx")

options_df = options_df.with_columns(
    (pl.col('divYieldSum') / np.e**(pl.col('timeToExp') * pl.col('dgs1'))).alias('discountDivYield')
).with_columns(
    (pl.col('base_close') * (1 - pl.col('discountDivYield'))).alias('discountBaseClose')
).with_columns(
    pl.when(pl.col('timeToExp') < 1).then(pl.col('discountBaseClose')).otherwise(pl.col('base_close')).alias('geekBaseClose'),
    pl.when(pl.col('timeToExp') < 1).then(0).otherwise(pl.col('divYieldSum')).alias('geekDivYieldSum')
)

In [24]:
# Может быть тут кроется рыночная неэффективность и в будущем нужно исследовать этот момент. Например пофиг на греки, использовать опционы только как профит-стоп лосс, 
# тогда я смогу включить эти данные в обучение.

options_df = options_df.filter(
    ~(
        (pl.col('ask') == 0) | (pl.col('bid') == 0)
    )
)

In [25]:
def calc_greeks(df: pl.DataFrame, func_to_calc: str, price: str) -> np.array:
    import py_vollib.black_scholes.greeks.numerical
    import py_vollib_vectorized

    func_to_calc = getattr(py_vollib_vectorized, func_to_calc)
    greek = func_to_calc(
        flag=df['right'].to_list(), 
        S=df['geekBaseClose'].to_list(), 
        K=df['strike'].to_list(), 
        t=df['timeToExp'].to_list(), 
        r=df['dgs1'].to_list(), 
        sigma=df[f'IV_{price}'].to_list(), 
        q=df['geekDivYieldSum'].to_list(),  # df['divYield'].to_list(),
        model='black_scholes_merton', 
        return_as='numpy'
    ) 
    
    return greek


def calc_iv_greeks(df: pl.DataFrame, price: str) -> pl.DataFrame:
    import py_vollib.black_scholes_merton.implied_volatility
    import py_vollib_vectorized
    
    iv = py_vollib_vectorized.vectorized_implied_volatility(
        price=df[price].to_list(), 
        S=df['geekBaseClose'].to_list(), 
        K=df['strike'].to_list(),
        t=df['timeToExp'].to_list(),
        r=df['dgs1'].to_list(),
        flag=df['right'].to_list(),
        q=df['geekDivYieldSum'].to_list(),  # df['divYield'].to_list(), 
        model='black_scholes_merton',
        return_as='numpy'
    )
    df = df.with_columns(pl.Series(iv).alias(f'IV_{price}'))

    delta = calc_greeks(df, 'vectorized_delta', price)
    gamma = calc_greeks(df, 'vectorized_gamma', price)
    theta = calc_greeks(df, 'vectorized_theta', price)
    vega = calc_greeks(df, 'vectorized_vega', price)
    rho = calc_greeks(df, 'vectorized_rho', price)
    df = df.with_columns(
        pl.Series(delta).alias(f'delta_{price}'),
        pl.Series(gamma).alias(f'gamma_{price}'),
        pl.Series(theta).alias(f'theta_{price}'),
        pl.Series(vega).alias(f'vega_{price}'),
        pl.Series(rho).alias(f'rho_{price}')
    )

    return df

In [26]:
%%time
options_df = calc_iv_greeks(options_df, 'ask')

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



CPU times: total: 39.1 s
Wall time: 56.7 s


In [27]:
%%time
options_df = calc_iv_greeks(options_df, 'bid')

IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



CPU times: total: 35.5 s
Wall time: 51.6 s


# Cleaning data for save

In [28]:
# Может быть тут кроется рыночная неэффективность и в будущем нужно исследовать этот момент. Например пофиг на греки, использовать опционы только как профит-стоп лосс, 
# тогда я смогу включить эти данные в обучение.

options_df = options_df.filter(
    ~(
        (pl.col('delta_ask').is_nan()) | (pl.col('delta_bid').is_nan())
    )
)

In [29]:
options_df = options_df.drop([
    'idx', 'divYieldSum', 'dgs1', 'discountDivYield', 'discountBaseClose', 'geekBaseClose', 'geekDivYieldSum',
    'bid_exchange', 'ask_exchange', 'bid_condition', 'ask_condition', 
    'ms_of_day'
])

options_df = options_df.with_columns(
    [pl.col(col).cast(pl.Float32) for col in options_df.columns if ('_ask' in col) or ('_bid' in col)]
)

In [30]:
options_df.drop(['date', 'expiration', 'ticker', 'right']).select(pl.all().is_nan().sum())

strike,ms_of_day2,open,high,low,close,volume,count,bid_size,bid,ask_size,ask,timeToExp,splitFactor,base_close,IV_ask,delta_ask,gamma_ask,theta_ask,vega_ask,rho_ask,IV_bid,delta_bid,gamma_bid,theta_bid,vega_bid,rho_bid
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [31]:
options_df.drop(['date', 'expiration', 'ticker', 'right']).select(pl.all().is_null().sum())

strike,ms_of_day2,open,high,low,close,volume,count,bid_size,bid,ask_size,ask,timeToExp,splitFactor,base_close,IV_ask,delta_ask,gamma_ask,theta_ask,vega_ask,rho_ask,IV_bid,delta_bid,gamma_bid,theta_bid,vega_bid,rho_bid
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [32]:
(options_df.drop(['date', 'expiration', 'ticker', 'right']) == 0).sum()

strike,ms_of_day2,open,high,low,close,volume,count,bid_size,bid,ask_size,ask,timeToExp,splitFactor,base_close,IV_ask,delta_ask,gamma_ask,theta_ask,vega_ask,rho_ask,IV_bid,delta_bid,gamma_bid,theta_bid,vega_bid,rho_bid
u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32,u32
0,4480261,4480262,4480261,4480261,4480262,4480261,4480279,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
# Add check for df[need_cols].select(pl.all().is_infinite()).sum()

# Add proba out-of-money

In [33]:
# In the future need correct to long-tail distribution

from scipy.stats import norm

options_df = options_df.join(treasury_y1, how='left', on='date')

options_df = options_df.with_columns(
    (
        ((pl.col('base_close') / pl.col('strike')).log() + (pl.col('dgs1') - pl.col('IV_ask')**2 / 2) * pl.col('timeToExp')) / 
        (pl.col('IV_ask') * np.sqrt(pl.col('timeToExp')))
    ).alias('d2_ask'),
    (
        ((pl.col('base_close') / pl.col('strike')).log() + (pl.col('dgs1') - pl.col('IV_bid')**2 / 2) * pl.col('timeToExp')) / 
        (pl.col('IV_bid') * np.sqrt(pl.col('timeToExp')))
    ).alias('d2_bid')
)

options_df = options_df.with_columns(
    (pl.when(pl.col('right') == 'p').then(norm.cdf(options_df['d2_ask'])).otherwise(1-norm.cdf(options_df['d2_ask']))).cast(pl.Float32).alias('probaOutOfM_ask'),
    (pl.when(pl.col('right') == 'p').then(norm.cdf(options_df['d2_bid'])).otherwise(1-norm.cdf(options_df['d2_bid']))).cast(pl.Float32).alias('probaOutOfM_bid'),
).drop(['d2_ask', 'd2_bid', 'dgs1'])

# Add ETF / Stock filter

In [34]:
path = r'G:\Биржа\Stocks. BigData\Цены\Дейли\tiingo\USA'

etf_list = []
for root, dirs, files in os.walk(os.path.join(path, 'ETF')):
    for file in files:
        etf_list.append(file.replace('.csv', ''))

stocks_list = []
for root, dirs, files in os.walk(os.path.join(path, 'Stock')):
    for file in files:
        stocks_list.append(file.replace('.csv', ''))

same_tickers = set(etf_list) & set(stocks_list)
etf_list = set(etf_list) - set(same_tickers)
stocks_list = set(stocks_list) - set(same_tickers)

tickers_df = pl.concat([
    pl.DataFrame(data={'ticker': list(etf_list), "type": ['ETF'] * len(etf_list)}),
    pl.DataFrame(data={'ticker': list(stocks_list), "type": ['Stock'] * len(stocks_list)}),
])

In [35]:
options_df = options_df.join(tickers_df, how='left', on='ticker')

# Save

In [38]:
options_df.filter(pl.col('type') == 'ETF').drop('type')\
    .write_parquet(f'data/options/with_greeks/{file_name}_etf_greeks.parquet')

In [24]:
options_df.filter(pl.col('type') != 'ETF').drop('type')\
    .write_parquet(f'data/options/with_greeks/{file_name}_stocks_greeks.parquet')

In [ ]:
file_name = 'SPY'